# Model building

## Imports

In [1]:
import pandas as pd
from itertools import product
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

pd.set_option("display.max_columns", None)

## Helper functions

### Preparing data for model input

In [2]:
def get_feature_cols(
    df: pd.DataFrame,
    use_adj: bool = False,
    include_missing: bool = False,
    include_failed: bool = False,
) -> list[str]:
    if use_adj:
        feature_cols = [c for c in df.columns if c.endswith("_adjscore")]
    else:    
        feature_cols = [c for c in df.columns if c.endswith("_score")]

    if include_missing:
        feature_cols += [c for c in df.columns if c.endswith("_missing")]

    if include_failed:
        feature_cols += [c for c in df.columns if c.endswith("_failed")]

    return feature_cols


def build_modelling_df(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_cols: list[str] | None = None,
) -> pd.DataFrame:
    if target_cols is None:
        target_cols = ["target_binary", "target_fighter", "target_multiclass"]

    keep_cols = ["STUDENT_ID", "Batch"] + feature_cols + target_cols
    return df[keep_cols].copy()

### Evaluation

In [3]:
def evaluate_binary_predictions(
    y_true: pd.Series,
    y_pred: pd.Series,
    y_score: pd.Series | None,
    *,
    setup_name: str,
    stage_name: str,
    split_name: str,
    model_name: str,
    n_features: int,
    rf_params: dict | None = None,
) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    row = {
        "setup": setup_name,
        "stage": stage_name,
        "split": split_name,
        "model": model_name,
        "n_features": int(n_features),
        "n_rows": int(len(y_true)),
        "positive_rate_true": float(y_true.mean()),
        "positive_rate_pred": float(pd.Series(y_pred).mean()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "roc_auc": float(roc_auc_score(y_true, y_score))
        if y_score is not None
        else None,
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_pos": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_weighted": float(
            f1_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    if rf_params is not None:
        row.update(rf_params)

    return row

### For binary classfication: comparison between dummy, logistic regression, and random forest

In [4]:
def make_model_dict() -> dict[str, Pipeline]:
    return {
        "Dummy": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]
        ),
        "Logistic Regression": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=2000, random_state=42)),
            ]
        ),
        "Random Forest Imputed": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=100,
                        random_state=42,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
        "Random Forest": Pipeline(
            [
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=100,
                        random_state=42,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
    }

def prepare_xy(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
) -> tuple[pd.DataFrame, pd.Series]:
    out = df.dropna(subset=[target_col]).copy()
    X = out[feature_cols]
    y = out[target_col].astype(int)
    return X, y


def filter_passers(df: pd.DataFrame) -> pd.DataFrame:
    return df[df["target_binary"] == 1].copy()

In [5]:
def run_binary_stage_to_df(
    train_df: pd.DataFrame,
    eval_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    *,
    setup_name: str,
    stage_name: str,
    split_name: str,
) -> pd.DataFrame:
    X_train, y_train = prepare_xy(train_df, feature_cols, target_col)
    X_eval, y_eval = prepare_xy(eval_df, feature_cols, target_col)

    model_dict = make_model_dict()

    rows = []

    for model_name, model in model_dict.items():
        fitted = clone(model)
        fitted.fit(X_train, y_train)

        y_pred = fitted.predict(X_eval)

        y_score = None
        if hasattr(fitted, "predict_proba"):
            y_score = fitted.predict_proba(X_eval)[:, 1]
        elif hasattr(fitted, "decision_function"):
            y_score = fitted.decision_function(X_eval)

        row = evaluate_binary_predictions(
            y_true=y_eval,
            y_pred=y_pred,
            y_score=y_score,
            setup_name=setup_name,
            stage_name=stage_name,
            split_name=split_name,
            model_name=model_name,
            n_features=len(feature_cols),
        )
        rows.append(row)

    return pd.DataFrame(rows)

def print_stage_results(results_df: pd.DataFrame) -> None:
    display_cols = [
        "setup",
        "stage",
        "split",
        "model",
        "n_rows",
        "n_features",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "tn",
        "fp",
        "fn",
        "tp",
    ]
    print(results_df[display_cols].round(5).to_string(index=False))

### Hyperparameter tuning for random forest

In [6]:
def build_rf_param_grid(param_grid: dict[str, list]) -> list[dict]:
    keys = list(param_grid.keys())
    values = [param_grid[k] for k in keys]
    return [dict(zip(keys, combo)) for combo in product(*values)]


def tune_random_forest_to_df(
    train_df: pd.DataFrame,
    eval_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    *,
    setup_name: str,
    stage_name: str,
    split_name: str,
    param_grid: dict[str, list] | None = None,
) -> pd.DataFrame:
    if param_grid is None:
        param_grid = {
            "n_estimators": [100, 300, 500],
            "max_depth": [None, 5, 10, 15],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 5],
        }

    X_train, y_train = prepare_xy(train_df, feature_cols, target_col)
    X_eval, y_eval = prepare_xy(eval_df, feature_cols, target_col)

    rows = []
    rf_param_list = build_rf_param_grid(param_grid)

    for rf_params in rf_param_list:
        pipe = Pipeline(
            [
                (
                    "model",
                    RandomForestClassifier(
                        random_state=42,
                        n_jobs=-1,
                        **rf_params,
                    ),
                ),
            ]
        )

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_eval)
        y_score = pipe.predict_proba(X_eval)[:, 1]

        row = evaluate_binary_predictions(
            y_true=y_eval,
            y_pred=y_pred,
            y_score=y_score,
            setup_name=setup_name,
            stage_name=stage_name,
            split_name=split_name,
            model_name="Random Forest",
            n_features=len(feature_cols),
            rf_params=rf_params,
        )
        rows.append(row)

    results_df = pd.DataFrame(rows)
    return results_df.sort_values(
        by=["f1_macro", "roc_auc", "accuracy"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    
    
def evaluate_single_rf_config_to_df(
    train_df: pd.DataFrame,
    eval_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    *,
    setup_name: str,
    stage_name: str,
    split_name: str,
    rf_params: dict,
) -> pd.DataFrame:
    X_train, y_train = prepare_xy(train_df, feature_cols, target_col)
    X_eval, y_eval = prepare_xy(eval_df, feature_cols, target_col)

    pipe = Pipeline(
        [
            (
                "model",
                RandomForestClassifier(
                    random_state=42,
                    n_jobs=-1,
                    **rf_params,
                ),
            ),
        ]
    )

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_eval)
    y_score = pipe.predict_proba(X_eval)[:, 1]

    row = evaluate_binary_predictions(
        y_true=y_eval,
        y_pred=y_pred,
        y_score=y_score,
        setup_name=setup_name,
        stage_name=stage_name,
        split_name=split_name,
        model_name="Random Forest",
        n_features=len(feature_cols),
        rf_params=rf_params,
    )

    return pd.DataFrame([row])

## Load data

In [7]:
df = pd.read_parquet("../data/03_clean/bwc_160_207.parquet")
df

,STUDENT_ID,GH 1_score,GH 1_adjscore,GH 1_failed,GH 1_missing,GH 2_score,GH 2_adjscore,GH 2_failed,GH 2_missing,GH 3_score,GH 3_adjscore,GH 3_failed,GH 3_missing,GH 4_score,GH 4_adjscore,GH 4_failed,GH 4_missing,GH 5_score,GH 5_adjscore,GH 5_failed,GH 5_missing,GH 6_score,GH 6_adjscore,GH 6_failed,GH 6_missing,GH 7_score,GH 7_adjscore,GH 7_failed,GH 7_missing,GH 8_score,GH 8_adjscore,GH 8_failed,GH 8_missing,GH 9_score,GH 9_adjscore,GH 9_failed,GH 9_missing,GH 10_score,GH 10_adjscore,GH 10_failed,GH 10_missing,GH 11_score,GH 11_adjscore,GH 11_failed,GH 11_missing,GH 12_score,GH 12_adjscore,GH 12_failed,GH 12_missing,GH 13_score,GH 13_adjscore,GH 13_failed,GH 13_missing,GH 14_score,GH 14_adjscore,GH 14_failed,GH 14_missing,GH 15_score,GH 15_adjscore,GH 15_failed,GH 15_missing,GH 17_score,GH 17_adjscore,GH 17_failed,GH 17_missing,GH 19_score,GH 19_adjscore,GH 19_failed,GH 19_missing,GH 21_score,GH 21_adjscore,GH 21_failed,GH 21_missing,GH 22_score,GH 22_adjscore,GH 22_failed,GH 22_missing,GH 24_score,GH 24_adjscore,GH 24_failed,GH 24_missing,GH 25_score,GH 25_adjscore,GH 25_failed,GH 25_missing,GH 27_score,GH 27_adjscore,GH 27_failed,GH 27_missing,GH 28_score,GH 28_adjscore,GH 28_failed,GH 28_missing,GH 29_score,GH 29_adjscore,GH 29_failed,GH 29_missing,GH 30_score,GH 30_adjscore,GH 30_failed,GH 30_missing,GH 31_score,GH 31_adjscore,GH 31_failed,GH 31_missing,GH 32_score,GH 32_adjscore,GH 32_failed,GH 32_missing,GH 33_score,GH 33_adjscore,GH 33_failed,GH 33_missing,GH 34_score,GH 34_adjscore,GH 34_failed,GH 34_missing,GHT_score,GHT_adjscore,GHT_failed,GHT_missing,IF 1_score,IF 1_adjscore,IF 1_failed,IF 1_missing,IF 2_score,IF 2_adjscore,IF 2_failed,IF 2_missing,IF 3_score,IF 3_adjscore,IF 3_failed,IF 3_missing,IF 4_score,IF 4_adjscore,IF 4_failed,IF 4_missing,IF 5_score,IF 5_adjscore,IF 5_failed,IF 5_missing,IF 6_score,IF 6_adjscore,IF 6_failed,IF 6_missing,IFT_score,IFT_adjscore,IFT_failed,IFT_missing,Batch,BWC_Status,Stream_Group,Stream_Unit,target_binary,target_multiclass,target_fighter
0,160CHANC,4.215324,4.215324,False,False,3.937536,3.937536,False,False,4.0,4.0,False,False,3.736023,3.736023,False,False,3.726859,3.726859,False,False,3.7472,3.7472,False,False,3.210893,1.0,True,False,2.673016,1.0,True,False,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,160,Fail,<NA>,<NA>,0,0,0
1,160CHANG,3.926446,3.926446,False,False,3.746286,3.746286,False,False,4.094832,4.094832,False,False,4.036967,4.036967,False,False,4.111887,4.111887,False,False,3.783657,3.783657,False,False,2.528977,2.528977,True,False,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,160,Fail,<NA>,<NA>,0,0,0
2,160CHOOC,4.243429,4.243429,False,False,4.37966,4.37966,False,False,4.674136,4.674136,False,False,4.110342,4.110342,False,False,3.962646,3.962646,False,False,3.928343,3.928343,False,False,4.0,4.0,False,False,4.107553,4.107553,False,False,4.304875,4.304875,False,False,4.3375,4.3375,False,False,4.40368

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 701 entries, 0 to 700
Columns: 156 entries, STUDENT_ID to target_fighter
dtypes: Float64(74), Int64(4), boolean(74), string(4)
memory usage: 615.9 KB


## Train val test split

In [9]:
# Use for model development
train_df = df[df["Batch"] <= 190].reset_index(drop=True)
val_df = df[(df["Batch"] > 190) & (df["Batch"] <= 200)].reset_index(drop=True)

# Held out set used for testing 
test_df = df[(df["Batch"] > 200)].dropna(subset="target_binary").reset_index(drop=True)

# print(train_df["target_binary"].value_counts(dropna=False))
# print(train_df["target_fighter"].value_counts(dropna=False))
# print(train_df["target_multiclass"].value_counts(dropna=False))
# print()
# print(val_df["target_binary"].value_counts(dropna=False))
# print(val_df["target_fighter"].value_counts(dropna=False))
# print(val_df["target_multiclass"].value_counts(dropna=False))
# print()
# print(test_df["target_binary"].value_counts(dropna=False))
# print(test_df["target_fighter"].value_counts(dropna=False))
# print(test_df["target_multiclass"].value_counts(dropna=False))

## 2-pass binary classification: pass vs fail BWC, fighter vs non-fighter for passes

### Feature selection

- For both stages, using scores only without zero imputation on random forest seems to work the best.

#### Setup 1: score only

In [40]:
score_only_cols = get_feature_cols(train_df)

In [41]:
# Evaluate on val

# Stage 1: pass vs fail bwc
train_score_only = build_modelling_df(train_df, score_only_cols)
val_score_only = build_modelling_df(val_df, score_only_cols)
setup1_stage1_val = run_binary_stage_to_df(
    train_df=train_score_only,
    eval_df=val_score_only,
    feature_cols=score_only_cols,
    target_col="target_binary",
    setup_name="Setup 1: score only",
    stage_name="Stage 1: pass/fail",
    split_name="val",
)

# Stage 2: among the bwc passes, fighter vs non-fighter
train_score_only_passers = filter_passers(train_score_only)
val_score_only_passers = filter_passers(val_score_only)
setup1_stage2_val = run_binary_stage_to_df(
    train_df=train_score_only_passers,
    eval_df=val_score_only_passers,
    feature_cols=score_only_cols,
    target_col="target_fighter",
    setup_name="Setup 1: score only",
    stage_name="Stage 2: fighter/non-fighter",
    split_name="val",
)

In [42]:
# Re-train on train + val, evaluate on test

# Stage 1: pass vs fail bwc
trainval_score_only = build_modelling_df(pd.concat([train_df, val_df], ignore_index=True), score_only_cols)
test_score_only = build_modelling_df(test_df, score_only_cols)
setup1_stage1_test = run_binary_stage_to_df(
    train_df=trainval_score_only,
    eval_df=test_score_only,
    feature_cols=score_only_cols,
    target_col="target_binary",
    setup_name="Setup 1: score only",
    stage_name="Stage 1: pass/fail",
    split_name="test",
)

# Stage 2: among the bwc passes, fighter vs non-fighter
trainval_score_only_passers = filter_passers(trainval_score_only)
test_score_only_passers = filter_passers(test_score_only)
setup1_stage2_test = run_binary_stage_to_df(
    train_df=trainval_score_only_passers,
    eval_df=test_score_only_passers,
    feature_cols=score_only_cols,
    target_col="target_fighter",
    setup_name="Setup 1: score only",
    stage_name="Stage 2: fighter/non-fighter",
    split_name="test",
)

#### Setup 2: adjusted score only

In [43]:
adjscore_only_cols = get_feature_cols(train_df, use_adj=True)

# Evaluate on val

# Stage 1: pass vs fail bwc
train_adjscore_only = build_modelling_df(train_df, adjscore_only_cols)
val_adjscore_only = build_modelling_df(val_df, adjscore_only_cols)
setup2_stage1_val = run_binary_stage_to_df(
    train_df=train_adjscore_only,
    eval_df=val_adjscore_only,
    feature_cols=adjscore_only_cols,
    target_col="target_binary",
    setup_name="Setup 2: adjscore only",
    stage_name="Stage 1: pass/fail",
    split_name="val",
)

# Stage 2: among the bwc passes, fighter vs non-fighter
train_adjscore_only_passers = filter_passers(train_adjscore_only)
val_adjscore_only_passers = filter_passers(val_adjscore_only)
setup2_stage2_val = run_binary_stage_to_df(
    train_df=train_adjscore_only_passers,
    eval_df=val_adjscore_only_passers,
    feature_cols=adjscore_only_cols,
    target_col="target_fighter",
    setup_name="Setup 2: adjscore only",
    stage_name="Stage 2: fighter/non-fighter",
    split_name="val",
)

# Re-train on train + val, evaluate on test

# Stage 1: pass vs fail bwc
trainval_adjscore_only = build_modelling_df(pd.concat([train_df, val_df], ignore_index=True), adjscore_only_cols)
test_adjscore_only = build_modelling_df(test_df, adjscore_only_cols)
setup2_stage1_test = run_binary_stage_to_df(
    train_df=trainval_adjscore_only,
    eval_df=test_adjscore_only,
    feature_cols=adjscore_only_cols,
    target_col="target_binary",
    setup_name="Setup 2: adjscore only",
    stage_name="Stage 1: pass/fail",
    split_name="test",
)

# Stage 2: among the bwc passes, fighter vs non-fighter
trainval_adjscore_only_passers = filter_passers(trainval_adjscore_only)
test_adjscore_only_passers = filter_passers(test_adjscore_only)
setup2_stage2_test = run_binary_stage_to_df(
    train_df=trainval_adjscore_only_passers,
    eval_df=test_adjscore_only_passers,
    feature_cols=adjscore_only_cols,
    target_col="target_fighter",
    setup_name="Setup 2: adjscore only",
    stage_name="Stage 2: fighter/non-fighter",
    split_name="test",
)

#### Setup 3: score + missing

In [44]:
score_missing_cols = get_feature_cols(train_df, include_missing=True)

# Evaluate on val

# Stage 1: pass vs fail bwc
train_score_missing = build_modelling_df(train_df, score_missing_cols)
val_score_missing = build_modelling_df(val_df, score_missing_cols)
setup3_stage1_val = run_binary_stage_to_df(
    train_df=train_score_missing,
    eval_df=val_score_missing,
    feature_cols=score_missing_cols,
    target_col="target_binary",
    setup_name="Setup 3: score + missing",
    stage_name="Stage 1: pass/fail",
    split_name="val",
)

# Stage 2: among the bwc passes, fighter vs non-fighter
train_score_missing_passers = filter_passers(train_score_missing)
val_score_missing_passers = filter_passers(val_score_missing)
setup3_stage2_val = run_binary_stage_to_df(
    train_df=train_score_missing_passers,
    eval_df=val_score_missing_passers,
    feature_cols=score_missing_cols,
    target_col="target_fighter",
    setup_name="Setup 3: score + missing",
    stage_name="Stage 2: fighter/non-fighter",
    split_name="val",
)

# Re-train on train + val, evaluate on test

# Stage 1: pass vs fail bwc
trainval_score_missing = build_modelling_df(pd.concat([train_df, val_df], ignore_index=True), score_missing_cols)
test_score_missing = build_modelling_df(test_df, score_missing_cols)
setup3_stage1_test = run_binary_stage_to_df(
    train_df=trainval_score_missing,
    eval_df=test_score_missing,
    feature_cols=score_missing_cols,
    target_col="target_binary",
    setup_name="Setup 3: score + missing",
    stage_name="Stage 1: pass/fail",
    split_name="test",
)

# Stage 2: among the bwc passes, fighter vs non-fighter
trainval_score_missing_passers = filter_passers(trainval_score_missing)
test_score_missing_passers = filter_passers(test_score_missing)
setup3_stage2_test = run_binary_stage_to_df(
    train_df=trainval_score_missing_passers,
    eval_df=test_score_missing_passers,
    feature_cols=score_missing_cols,
    target_col="target_fighter",
    setup_name="Setup 3: score + missing",
    stage_name="Stage 2: fighter/non-fighter",
    split_name="test",
)

#### Setup 4: score + fail

In [45]:
score_fail_cols = get_feature_cols(train_df, include_missing=True)

# Evaluate on val

# Stage 1: pass vs fail bwc
train_score_fail = build_modelling_df(train_df, score_fail_cols)
val_score_fail = build_modelling_df(val_df, score_fail_cols)
setup4_stage1_val = run_binary_stage_to_df(
    train_df=train_score_fail,
    eval_df=val_score_fail,
    feature_cols=score_fail_cols,
    target_col="target_binary",
    setup_name="Setup 4: score + fail",
    stage_name="Stage 1: pass/fail",
    split_name="val",
)

# Stage 2: among the bwc passes, fighter vs non-fighter
train_score_fail_passers = filter_passers(train_score_fail)
val_score_fail_passers = filter_passers(val_score_fail)
setup4_stage2_val = run_binary_stage_to_df(
    train_df=train_score_fail_passers,
    eval_df=val_score_fail_passers,
    feature_cols=score_fail_cols,
    target_col="target_fighter",
    setup_name="Setup 4: score + fail",
    stage_name="Stage 2: fighter/non-fighter",
    split_name="val",
)

# Re-train on train + val, evaluate on test

# Stage 1: pass vs fail bwc
trainval_score_fail = build_modelling_df(pd.concat([train_df, val_df], ignore_index=True), score_fail_cols)
test_score_fail = build_modelling_df(test_df, score_fail_cols)
setup4_stage1_test = run_binary_stage_to_df(
    train_df=trainval_score_fail,
    eval_df=test_score_fail,
    feature_cols=score_fail_cols,
    target_col="target_binary",
    setup_name="Setup 4: score + fail",
    stage_name="Stage 1: pass/fail",
    split_name="test",
)

# Stage 2: among the bwc passes, fighter vs non-fighter
trainval_score_fail_passers = filter_passers(trainval_score_fail)
test_score_fail_passers = filter_passers(test_score_fail)
setup4_stage2_test = run_binary_stage_to_df(
    train_df=trainval_score_fail_passers,
    eval_df=test_score_fail_passers,
    feature_cols=score_fail_cols,
    target_col="target_fighter",
    setup_name="Setup 4: score + fail",
    stage_name="Stage 2: fighter/non-fighter",
    split_name="test",
)

#### Setup 5: score + missing + fail

In [46]:
score_missing_fail_cols = get_feature_cols(train_df, include_missing=True)

# Evaluate on val

# Stage 1: pass vs fail bwc
train_score_missing_fail = build_modelling_df(train_df, score_missing_fail_cols)
val_score_missing_fail = build_modelling_df(val_df, score_missing_fail_cols)
setup5_stage1_val = run_binary_stage_to_df(
    train_df=train_score_missing_fail,
    eval_df=val_score_missing_fail,
    feature_cols=score_missing_fail_cols,
    target_col="target_binary",
    setup_name="Setup 5: score + fail + missing",
    stage_name="Stage 1: pass/fail",
    split_name="val",
)

# Stage 2: among the bwc passes, fighter vs non-fighter
train_score_missing_fail_passers = filter_passers(train_score_missing_fail)
val_score_missing_fail_passers = filter_passers(val_score_missing_fail)
setup5_stage2_val = run_binary_stage_to_df(
    train_df=train_score_missing_fail_passers,
    eval_df=val_score_missing_fail_passers,
    feature_cols=score_missing_fail_cols,
    target_col="target_fighter",
    setup_name="Setup 5: score + fail + missing",
    stage_name="Stage 2: fighter/non-fighter",
    split_name="val",
)

# Re-train on train + val, evaluate on test

# Stage 1: pass vs fail bwc
trainval_score_missing_fail = build_modelling_df(pd.concat([train_df, val_df], ignore_index=True), score_missing_fail_cols)
test_score_missing_fail = build_modelling_df(test_df, score_missing_fail_cols)
setup5_stage1_test = run_binary_stage_to_df(
    train_df=trainval_score_missing_fail,
    eval_df=test_score_missing_fail,
    feature_cols=score_missing_fail_cols,
    target_col="target_binary",
    setup_name="Setup 5: score + fail + missing",
    stage_name="Stage 1: pass/fail",
    split_name="test",
)

# Stage 2: among the bwc passes, fighter vs non-fighter
trainval_score_missing_fail_passers = filter_passers(trainval_score_missing_fail)
test_score_missing_fail_passers = filter_passers(test_score_missing_fail)
setup5_stage2_test = run_binary_stage_to_df(
    train_df=trainval_score_missing_fail_passers,
    eval_df=test_score_missing_fail_passers,
    feature_cols=score_missing_fail_cols,
    target_col="target_fighter",
    setup_name="Setup 5: score + fail + missing",
    stage_name="Stage 2: fighter/non-fighter",
    split_name="test",
)

#### Results

In [47]:
results_df = pd.concat(
    [
        setup1_stage1_val,
        setup1_stage1_test,
        setup1_stage2_val,
        setup1_stage2_test,
        setup2_stage1_val,
        setup2_stage1_test,
        setup2_stage2_val,
        setup2_stage2_test,
        setup3_stage1_val,
        setup3_stage1_test,
        setup3_stage2_val,
        setup3_stage2_test,
        setup4_stage1_val,
        setup4_stage1_test,
        setup4_stage2_val,
        setup4_stage2_test,
        setup5_stage1_val,
        setup5_stage1_test,
        setup5_stage2_val,
        setup5_stage2_test,
        
    ],
    ignore_index=True,
)

results_df = results_df.sort_values(
    by=["setup", "stage", "split", "model"]
).reset_index(drop=True)

##### Stage 1: pass vs fail BWC

In [48]:
results_df[(results_df["split"] == "val") & (results_df["stage"]=="Stage 1: pass/fail")].sort_values(by="f1_macro", ascending=False).head(10)

,setup,stage,split,model,n_features,n_rows,positive_rate_true,positive_rate_pred,accuracy,roc_auc,precision,recall,f1_pos,f1_macro,f1_weighted,tn,fp,fn,tp
7,Setup 1: score only,Stage 1: pass/fail,val,Random Forest Imputed,37,157,0.573248,0.630573,0.917197,0.933416,0.888889,0.977778,0.931217,0.913608,0.916188,56,11,2,88
54,Setup 4: score + fail,Stage 1: pass/fail,val,Random Forest,74,157,0.573248,0.636943,0.910828,0.944444,0.880000,0.977778,0.926316,0.906706,0.909579,55,12,2,88
39,Setup 3: score + missing,Stage 1: pass/fail,val,Random Forest Imputed,74,157,0.573248,0.636943,0.910828,0.945274,0.880000,0.977778,0.926316,0.906706,0.909579,55,12,2,88
38,Setup 3: score + missing,Stage 1: pass/fail,val,Random Forest,74,157,0.573248,0.636943,0.910828,0.944444,0.880000,0.977778,0.926316,0.906706,0.909579,55,12,2,88
71,Setup 5: score + fail + missing,Stage 1: pass/fail,val,Random Forest Imputed,74,157,0.573248,0.636943,0.910828,0.945274,0.880000,0.977778,0.926316,0.906706,0.909579,55,12,2,88
70,Setup 5: score + fail + missing,Stage 1: pass/fail,val,Random Forest,74,157,0.573248,0.636943,0.910828,0.944444,0.880000,0.977778,0.926316,0.906706,0.909579,55,12,2,88
55,Setup 4: score + fail,Stage 1: pass/fail,val,Random Forest Imputed,74,157,0.573248,0.636943,0.910828,0.945274,0.880000,0.977778,0.926316,0.906706,0.909579,55,12,2,88
22,Setup 2: adjscore only,Stage 1: pass/fail,val,Random Forest,37,157,0.573248,0.592357,0.904459,0.956882,0.903226,0.933333,0.918033,0.901764,0.904148,58,9,6,84
23,Setup 2: adjscore only,Stage 1: pass/fail,val,Random Forest Imputed,37,157,0.573248,0.605096,0.904459,0.937396,0.894737,0.944444,0.918919,0.901320,0.903898,57,10,5,85
6,Setup 1: score only,Stage 1: pass/fail,val,Random Forest,37,157,0.573248,0.643312,0.904459,0.933748,0.871287,0.977778,0.921466,0.899757,0.902938,54,13,2,88


In [49]:
results_df[(results_df["split"] == "test") & (results_df["stage"]=="Stage 1: pass/fail")].sort_values(by="f1_macro", ascending=False).head(10)

,setup,stage,split,model,n_features,n_rows,positive_rate_true,positive_rate_pred,accuracy,roc_auc,precision,recall,f1_pos,f1_macro,f1_weighted,tn,fp,fn,tp
2,Setup 1: score only,Stage 1: pass/fail,test,Random Forest,37,56,0.75,0.732143,0.946429,0.977891,0.975610,0.952381,0.963855,0.930204,0.947029,13,1,2,40
67,Setup 5: score + fail + missing,Stage 1: pass/fail,test,Random Forest Imputed,74,56,0.75,0.767857,0.946429,0.937925,0.953488,0.976190,0.964706,0.926797,0.945752,12,2,1,41
35,Setup 3: score + missing,Stage 1: pass/fail,test,Random Forest Imputed,74,56,0.75,0.767857,0.946429,0.937925,0.953488,0.976190,0.964706,0.926797,0.945752,12,2,1,41
18,Setup 2: adjscore only,Stage 1: pass/fail,test,Random Forest,37,56,0.75,0.767857,0.946429,0.957483,0.953488,0.976190,0.964706,0.926797,0.945752,12,2,1,41
51,Setup 4: score + fail,Stage 1: pass/fail,test,Random Forest Imputed,74,56,0.75,0.767857,0.946429,0.937925,0.953488,0.976190,0.964706,0.926797,0.945752,12,2,1,41
19,Setup 2: adjscore only,Stage 1: pass/fail,test,Random Forest Imputed,37,56,0.75,0.714286,0.928571,0.971939,0.975000,0.928571,0.951220,0.908943,0.930081,13,1,3,39
50,Setup 4: score + fail,Stage 1: pass/fail,test,Random Forest,74,56,0.75,0.750000,0.928571,0.960034,0.952381,0.952381,0.952381,0.904762,0.928571,12,2,2,40
66,Setup 5: score + fail + missing,Stage 1: pass/fail,test,Random Forest,74,56,0.75,0.750000,0.928571,0.960034,0.952381,0.952381,0.952381,0.904762,0.928571,12,2,2,40
34,Setup 3: score + missing,Stage 1: pass/fail,test,Random Forest,74,56,0.75,0.750000,0.928571,0.960034,0.952381,0.952381,0.952381,0.904762,0.928571,12,2,2,40
3,Setup 1: score only,Stage 1: pass/fail,test,Random Forest Imputed,37,56,0.75,0.714286,0.892857,0.950680,0.950000,0.904762,0.926829,0.863415,0.895122,12,2,4,38


##### Stage 2: among the BWC passes, fighter vs non-fighter

In [50]:
results_df[(results_df["split"] == "val") & (results_df["stage"]=="Stage 2: fighter/non-fighter")].sort_values(by="f1_macro", ascending=False).head(10)

,setup,stage,split,model,n_features,n_rows,positive_rate_true,positive_rate_pred,accuracy,roc_auc,precision,recall,f1_pos,f1_macro,f1_weighted,tn,fp,fn,tp
14,Setup 1: score only,Stage 2: fighter/non-fighter,val,Random Forest,37,90,0.522222,0.466667,0.655556,0.701385,0.690476,0.617021,0.651685,0.655513,0.655343,30,13,18,29
79,Setup 5: score + fail + missing,Stage 2: fighter/non-fighter,val,Random Forest Imputed,74,90,0.522222,0.533333,0.655556,0.715982,0.666667,0.680851,0.673684,0.654489,0.655342,27,16,15,32
47,Setup 3: score + missing,Stage 2: fighter/non-fighter,val,Random Forest Imputed,74,90,0.522222,0.533333,0.655556,0.715982,0.666667,0.680851,0.673684,0.654489,0.655342,27,16,15,32
31,Setup 2: adjscore only,Stage 2: fighter/non-fighter,val,Random Forest Imputed,37,90,0.522222,0.533333,0.655556,0.665017,0.666667,0.680851,0.673684,0.654489,0.655342,27,16,15,32
63,Setup 4: score + fail,Stage 2: fighter/non-fighter,val,Random Forest Imputed,74,90,0.522222,0.533333,0.655556,0.715982,0.666667,0.680851,0.673684,0.654489,0.655342,27,16,15,32
62,Setup 4: score + fail,Stage 2: fighter/non-fighter,val,Random Forest,74,90,0.522222,0.522222,0.644444,0.681346,0.659574,0.659574,0.659574,0.643741,0.644444,27,16,16,31
46,Setup 3: score + missing,Stage 2: fighter/non-fighter,val,Random Forest,74,90,0.522222,0.522222,0.644444,0.681346,0.659574,0.659574,0.659574,0.643741,0.644444,27,16,16,31
78,Setup 5: score + fail + missing,Stage 2: fighter/non-fighter,val,Random Forest,74,90,0.522222,0.522222,0.644444,0.681346,0.659574,0.659574,0.659574,0.643741,0.644444,27,16,16,31
30,Setup 2: adjscore only,Stage 2: fighter/non-fighter,val,Random Forest,37,90,0.522222,0.455556,0.622222,0.682335,0.658537,0.574468,0.613636,0.622036,0.621662,29,14,20,27
15,Setup 1: score only,Stage 2: fighter/non-fighter,val,Random Forest Imputed,37,90,0.522222,0.522222,0.600000,0.675903,0.617021,0.617021,0.617021,0.599208,0.600000,25,18,18,29


In [51]:
results_df[(results_df["split"] == "test") & (results_df["stage"]=="Stage 2: fighter/non-fighter")].sort_values(by="f1_macro", ascending=False).head(10)

,setup,stage,split,model,n_features,n_rows,positive_rate_true,positive_rate_pred,accuracy,roc_auc,precision,recall,f1_pos,f1_macro,f1_weighted,tn,fp,fn,tp
10,Setup 1: score only,Stage 2: fighter/non-fighter,test,Random Forest,37,42,0.571429,0.666667,0.714286,0.817130,0.714286,0.833333,0.769231,0.697115,0.707418,10,8,4,20
74,Setup 5: score + fail + missing,Stage 2: fighter/non-fighter,test,Random Forest,74,42,0.571429,0.714286,0.714286,0.804398,0.700000,0.875000,0.777778,0.688889,0.701587,9,9,3,21
58,Setup 4: score + fail,Stage 2: fighter/non-fighter,test,Random Forest,74,42,0.571429,0.714286,0.714286,0.804398,0.700000,0.875000,0.777778,0.688889,0.701587,9,9,3,21
42,Setup 3: score + missing,Stage 2: fighter/non-fighter,test,Random Forest,74,42,0.571429,0.714286,0.714286,0.804398,0.700000,0.875000,0.777778,0.688889,0.701587,9,9,3,21
26,Setup 2: adjscore only,Stage 2: fighter/non-fighter,test,Random Forest,37,42,0.571429,0.571429,0.666667,0.789352,0.708333,0.708333,0.708333,0.659722,0.666667,11,7,7,17
27,Setup 2: adjscore only,Stage 2: fighter/non-fighter,test,Random Forest Imputed,37,42,0.571429,0.619048,0.666667,0.729167,0.692308,0.750000,0.720000,0.654118,0.663529,10,8,6,18
11,Setup 1: score only,Stage 2: fighter/non-fighter,test,Random Forest Imputed,37,42,0.571429,0.714286,0.666667,0.721065,0.666667,0.833333,0.740741,0.637037,0.651852,8,10,4,20
75,Setup 5: score + fail + missing,Stage 2: fighter/non-fighter,test,Random Forest Imputed,74,42,0.571429,0.619048,0.619048,0.740741,0.653846,0.708333,0.680000,0.604706,0.615462,9,9,7,17
59,Setup 4: score + fail,Stage 2: fighter/non-fighter,test,Random Forest Imputed,74,42,0.571429,0.619048,0.619048,0.740741,0.653846,0.708333,0.680000,0.604706,0.615462,9,9,7,17
43,Setup 3: score + missing,Stage 2: fighter/non-fighter,test,Random Forest Imputed,74,42,0.571429,0.619048,0.619048,0.740741,0.653846,0.708333,0.680000,0.604706,0.615462,9,9,7,17


### Hyperparameter tuning

|stage| split | model         | n_features | n_rows | positive_rate_true | positive_rate_pred | accuracy | roc_auc  | precision | recall   | f1_pos   | f1_macro | f1_weighted | tn | fp | fn | tp |
|-----|-------|---------------|------------|--------|---------------------|---------------------|----------|----------|-----------|----------|----------|----------|--------------|----|----|----|----|
|1| val   | Random Forest | 37         | 157    | 0.573248            | 0.636943            | 0.923567 | 0.937479 | 0.890000  | 0.988889 | 0.936842 | 0.920034 | 0.922496     | 56 | 11 | 1  | 89 |
|1| test  | Random Forest | 37         | 56     | 0.750000            | 0.732143            | 0.946429 | 0.960884 | 0.975610  | 0.952381 | 0.963855 | 0.930204 | 0.947029     | 13 | 1  | 2  | 40 |
|2| val   | Random Forest | 37         | 90     | 0.522222            | 0.544444            | 0.688889 | 0.727857 | 0.693878  | 0.723404 | 0.708333 | 0.687500 | 0.688426     | 28 | 15 | 13 | 34 |
|2| test  | Random Forest | 37         | 42     | 0.571429            | 0.642857            | 0.738095 | 0.803241 | 0.740741  | 0.833333 | 0.784314 | 0.725490 | 0.733894     | 11 | 7  | 4  | 20 |


```Python
stage1_best_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

stage2_best_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=2,
    class_weight="balanced"
    random_state=42,
    n_jobs=-1
)
```

In [10]:
rf_param_grid = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5, 10],
    "class_weight": [None, "balanced"]
}

In [15]:
score_only_cols = get_feature_cols(train_df)

train_score_only = build_modelling_df(train_df, score_only_cols)
val_score_only = build_modelling_df(val_df, score_only_cols)
trainval_score_only = build_modelling_df(pd.concat([train_df, val_df], ignore_index=True), score_only_cols)
test_score_only = build_modelling_df(test_df, score_only_cols)

train_score_only_passers = filter_passers(train_score_only)
val_score_only_passers = filter_passers(val_score_only)
trainval_score_only_passers = filter_passers(trainval_score_only)
test_score_only_passers = filter_passers(test_score_only)


In [ ]:
stage1_rf_tuning_val = tune_random_forest_to_df(
    train_df=train_score_only,
    eval_df=val_score_only,
    feature_cols=score_only_cols,
    target_col="target_binary",
    setup_name="Setup 1: score only",
    stage_name="Stage 1: pass/fail",
    split_name="val",
    param_grid=rf_param_grid,
)

running_list = [stage1_rf_tuning_val.iloc[:5]]


for i in range(5):
    best_stage1_params = (
        stage1_rf_tuning_val.iloc[i][
            ["n_estimators", "max_depth", "min_samples_split", "min_samples_leaf", "class_weight"]
        ]
        .astype("Int64")
        .to_dict()
    )

    stage1_rf_best_test = evaluate_single_rf_config_to_df(
        train_df=trainval_score_only,
        eval_df=test_score_only,
        feature_cols=score_only_cols,
        target_col="target_binary",
        setup_name="Setup 1: score only",
        stage_name="Stage 1: pass/fail",
        split_name="test",
        rf_params=best_stage1_params,
    )

    running_list.append(stage1_rf_best_test)


rf_tuning_results_df_stage1_setup1 = pd.concat(
    running_list,
    ignore_index=True,
)

rf_tuning_results_df_stage1_setup1


,setup,stage,split,model,n_features,n_rows,positive_rate_true,positive_rate_pred,accuracy,roc_auc,precision,recall,f1_pos,f1_macro,f1_weighted,tn,fp,fn,tp,n_estimators,max_depth,min_samples_split,min_samples_leaf,class_weight
0,Setup 1: score only,Stage 1: pass/fail,val,Random Forest,37,157,0.573248,0.636943,0.923567,0.937479,0.890000,0.988889,0.936842,0.920034,0.922496,56,11,1,89,500,10.0,5,1,balanced
1,Setup 1: score only,Stage 1: pass/fail,val,Random Forest,37,157,0.573248,0.630573,0.917197,0.940299,0.888889,0.977778,0.931217,0.913608,0.916188,56,11,2,88,400,10.0,5,1,balanced
2,Setup 1: score only,Stage 1: pass/fail,val,Random Forest,37,157,0.573248,0.630573,0.917197,0.939801,0.888889,0.977778,0.931217,0.913608,0.916188,56,11,2,88,300,NaN,2,1,NaN
3,Setup 1: score only,Stage 1: pass/fail,val,Random Forest,37,157,0.573248,0.630573,0.917197,0.939386,0.888889,0.977778,0.931217,0.913608,0.916188,56,11,2,88,200,15.0,2,1,NaN
4,Setup 1: score only,Stage 1: pass/fail,val,Random Forest,37,157,0.573248,0.630573,0.917197,0.939303,0.888889,0.977778,0.931217,0.913608,0.916188,56,11,2,88,200,10.0,2,1,NaN
5,Setup 1: score only,Stage 1: pass/fail,test,Random Forest,37,56,0.750000,0.732143,0.946429,0.960884,0.975610,0.952381,0.963855,0.930204,0.947029,13,1,2,40,500,10,5,1,NaN
6,Setup 1: score only,Stage 1: pass/fail,test,Random Forest,37,56,0.750000,0.732143,0.946429,0.960884,0.975610,0.952381,0.963855,0.930204,0.947029,13,1,2,40,400,10,5,1,NaN
7,Setup 1: score only,Stage 1: pass/fail,test,Random Forest,37,56,0.750000,0.732143,0.946429,0.969388,0.975610,0.952381,0.963855,0.930204,0.947029,13,1,2,40,300,None,2,1,NaN
8,Setup 1: score only,Stage 1: pass/fail,test,Random Forest,37,56,0.750000,0.732143,0.946429,0.972789,0.975610,0.952381,0.963855,0.930204,0.947029,13,1,2,40,200,15,2,1,NaN
9,Setup 1: score only,Stage 1: pass/fail,test,Random Forest,37,56,0.750000,0.732143,0.910714,0.971088,0.951220,0.928571,0.939759,0.883673,0.911716,12,2,3,39,200,10,2,1,NaN


In [16]:
stage2_rf_tuning_val = tune_random_forest_to_df(
    train_df=train_score_only_passers,
    eval_df=val_score_only_passers,
    feature_cols=score_only_cols,
    target_col="target_fighter",
    setup_name="Setup 1: score only",
    stage_name="Stage 2: fighter/non-fighter",
    split_name="val",
    param_grid=rf_param_grid,
)

running_list = [stage2_rf_tuning_val.iloc[:5]]

for i in range(5):
    best_stage2_params = (
        stage2_rf_tuning_val.iloc[i][
            ["n_estimators", "max_depth", "min_samples_split", "min_samples_leaf", "class_weight"]
        ]
        .astype("Int64")
        .to_dict()
    )

    stage2_rf_best_test = evaluate_single_rf_config_to_df(
        train_df=trainval_score_only_passers,
        eval_df=test_score_only_passers,
        feature_cols=score_only_cols,
        target_col="target_fighter",
        setup_name="Setup 1: score only",
        stage_name="Stage 2: fighter/non-fighter",
        split_name="test",
        rf_params=best_stage2_params,
    )

    running_list.append(stage2_rf_best_test)


rf_tuning_results_df_stage2_setup1 = pd.concat(
    running_list,
    ignore_index=True,
)

rf_tuning_results_df_stage2_setup1

,setup,stage,split,model,n_features,n_rows,positive_rate_true,positive_rate_pred,accuracy,roc_auc,precision,recall,f1_pos,f1_macro,f1_weighted,tn,fp,fn,tp,n_estimators,max_depth,min_samples_split,min_samples_leaf,class_weight
0,Setup 1: score only,Stage 2: fighter/non-fighter,val,Random Forest,37,90,0.522222,0.544444,0.688889,0.727857,0.693878,0.723404,0.708333,0.687500,0.688426,28,15,13,34,100,10.0,10,2,balanced
1,Setup 1: score only,Stage 2: fighter/non-fighter,val,Random Forest,37,90,0.522222,0.544444,0.688889,0.717467,0.693878,0.723404,0.708333,0.687500,0.688426,28,15,13,34,100,10.0,10,2,NaN
2,Setup 1: score only,Stage 2: fighter/non-fighter,val,Random Forest,37,90,0.522222,0.533333,0.677778,0.721425,0.687500,0.702128,0.694737,0.676780,0.677578,28,15,14,33,100,NaN,10,2,balanced
3,Setup 1: score only,Stage 2: fighter/non-fighter,val,Random Forest,37,90,0.522222,0.533333,0.677778,0.721425,0.687500,0.702128,0.694737,0.676780,0.677578,28,15,14,33,100,15.0,10,2,balanced
4,Setup 1: score only,Stage 2: fighter/non-fighter,val,Random Forest,37,90,0.522222,0.533333,0.677778,0.709550,0.687500,0.702128,0.694737,0.676780,0.677578,28,15,14,33,300,10.0,2,1,balanced
5,Setup 1: score only,Stage 2: fighter/non-fighter,test,Random Forest,37,42,0.571429,0.642857,0.738095,0.803241,0.740741,0.833333,0.784314,0.725490,0.733894,11,7,4,20,100,10,10,2,NaN
6,Setup 1: score only,Stage 2: fighter/non-fighter,test,Random Forest,37,42,0.571429,0.642857,0.738095,0.803241,0.740741,0.833333,0.784314,0.725490,0.733894,11,7,4,20,100,10,10,2,NaN
7,Setup 1: score only,Stage 2: fighter/non-fighter,test,Random Forest,37,42,0.571429,0.642857,0.690476,0.800926,0.703704,0.791667,0.745098,0.675579,0.685511,10,8,5,19,100,None,10,2,NaN
8,Setup 1: score only,Stage 2: fighter/non-fighter,test,Random Forest,37,42,0.571429,0.642857,0.690476,0.798611,0.703704,0.791667,0.745098,0.675579,0.685511,10,8,5,19,100,15,10,2,NaN
9,Setup 1: score only,Stage 2: fighter/non-fighter,test,Random Forest,37,42,0.571429,0.690476,0.690476,0.800926,0.689655,0.833333,0.754717,0.667681,0.680115,9,9,4,20,300,10,2,1,NaN


## 1-pass 4-class multiclassification

In [29]:
score_only_cols = get_feature_cols(train_df)

train_score_only = build_modelling_df(train_df, score_only_cols)
val_score_only = build_modelling_df(val_df, score_only_cols)
trainval_score_only = build_modelling_df(pd.concat([train_df, val_df], ignore_index=True), score_only_cols)
test_score_only = build_modelling_df(test_df, score_only_cols)

train_score_only_passers = filter_passers(train_score_only)
val_score_only_passers = filter_passers(val_score_only)
trainval_score_only_passers = filter_passers(trainval_score_only)
test_score_only_passers = filter_passers(test_score_only)

In [ ]:
X = trainval_score_only[score_only_cols]
y = trainval_score_only["target_multiclass"]

model = RandomForestClassifier(
    n_estimators=500, random_state=42, n_jobs=-1, class_weight="balanced"
)

model.fit(X, y)

X_eval = test_score_only[score_only_cols]
y_eval = test_score_only["target_multiclass"]
y_pred = model.predict(X_eval)
y_score = model.predict_proba(X_eval)[:, 1]

print(
    classification_report(
        y_eval,
        y_pred,
        zero_division=0,
        digits=4,
        target_names=["fail BWC", "Fighter", "Transport", "Heli"],
    )
)


              precision    recall  f1-score   support

    fail BWC    0.68421   0.92857   0.78788        14
     Fighter    0.63333   0.79167   0.70370        24
   Transport    0.00000   0.00000   0.00000         7
        Heli    0.28571   0.18182   0.22222        11

    accuracy                        0.60714        56
   macro avg    0.40081   0.47551   0.42845        56
weighted avg    0.49860   0.60714   0.54221        56



## Best Model

In [12]:
score_only_cols = get_feature_cols(train_df)

trainval_score_only = build_modelling_df(pd.concat([train_df, val_df], ignore_index=True), score_only_cols)
# rename columns
trainval_score_only = trainval_score_only.rename(
    columns=lambda c: c.replace("_score", "")
)
trainval_score_only_passers = filter_passers(trainval_score_only)

test_score_only = build_modelling_df(test_df, score_only_cols)
test_score_only = test_score_only.rename(
    columns=lambda c: c.replace("_score", "")
)
test_score_only_passers = filter_passers(test_score_only)

cols_to_drop = [
    "STUDENT_ID",
    "Batch",
    "target_binary",
    "target_fighter",
    "target_multiclass",
]

In [13]:
X1 = trainval_score_only.drop(columns=cols_to_drop)
y1 = trainval_score_only["target_binary"]
X_eval = test_score_only.drop(columns=cols_to_drop)
y_eval = test_score_only["target_binary"]

stage1_best_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

stage1_best_model.fit(X1, y1)

y_pred1 = stage1_best_model.predict(X_eval)
y_score1 = stage1_best_model.predict_proba(X_eval)[:, 1]
print(
    classification_report(
        y_eval,
        y_pred1,
        zero_division=0,
        digits=4,
    )
)
print(f"ROC-AUC: {roc_auc_score(y_eval, y_score1):.4f}")

              precision    recall  f1-score   support

         0.0     0.9286    0.9286    0.9286        14
         1.0     0.9762    0.9762    0.9762        42

    accuracy                         0.9643        56
   macro avg     0.9524    0.9524    0.9524        56
weighted avg     0.9643    0.9643    0.9643        56

ROC-AUC: 0.9609


In [15]:
X2 = trainval_score_only_passers.drop(columns=cols_to_drop)
y2 = trainval_score_only_passers["target_fighter"]
X_eval = test_score_only_passers.drop(columns=cols_to_drop)
y_eval = test_score_only_passers["target_fighter"]

stage2_best_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

stage2_best_model.fit(X2, y2)

y_pred2 = stage2_best_model.predict(X_eval)
y_score2 = stage2_best_model.predict_proba(X_eval)[:, 1]
print(
    classification_report(
        y_eval,
        y_pred2,
        zero_division=0,
        digits=4,
    )
)
print(f"ROC-AUC: {roc_auc_score(y_eval, y_score2):.4f}")

              precision    recall  f1-score   support

         0.0     0.6923    0.5000    0.5806        18
         1.0     0.6897    0.8333    0.7547        24

    accuracy                         0.6905        42
   macro avg     0.6910    0.6667    0.6677        42
weighted avg     0.6908    0.6905    0.6801        42

ROC-AUC: 0.8009


In [16]:
# save
joblib.dump(stage1_best_model, "../models/stage1_best_model.joblib")
joblib.dump(stage2_best_model, "../models/stage2_best_model.joblib")

['../models/stage2_best_model.joblib']